<a href="https://colab.research.google.com/github/Luoxiaowanzi/Class/blob/main/Assignments/assignment_12_bayes_svm_neural.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Assignment 12: Predicting Hotel Booking Cancellations**
## Models: Naïve Bayes, Support Vector Machine (SVM), and Neural Network

**Objectives:**
- Understand how to use classification models (Naïve Bayes, SVM, Neural Networks) to predict hotel cancellations.
- Compare models in terms of accuracy, complexity, and business relevance.
- Interpret and communicate model results from a business perspective.

## **Business Context - Hotel Bookings**

You work as a data analyst for a hospitality group that manages both **Resort** and **City Hotels**. One major challenge in operations is the unpredictability of **booking cancellations**, which affects staffing, inventory, and revenue planning.

You’ve been asked to use historical booking data to predict whether a future booking will be canceled. Your insights will help management plan more effectively.


Your task is to:
1. Build and evaluate three models: Naïve Bayes, SVM, and Neural Network.
2. Compare performance.
3. Recommend which model is best suited for the business needs.

### Key Use Cases
- Understand customer booking behavior
- Explore factors related to cancellations
- Segment guests based on booking characteristics
- Compare city vs. resort hotel performance

<a href="https://colab.research.google.com/github/vandanara/UofUtah_IS4487/blob/main/Assignments/assignment_12_bayes_svm_neural.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


## Data Dictionary

This dataset contains booking information for two types of hotels: a **city hotel** and a **resort hotel**. Each record corresponds to a single booking and includes various details about the reservation, customer demographics, booking source, and whether the booking was canceled.

**Source**: [GitHub - TidyTuesday: Hotel Bookings](https://github.com/rfordatascience/tidytuesday/blob/master/data/2020/2020-02-11/readme.md)



### Data Dictionary

| Variable | Type | Description |
|----------|------|-------------|
| `hotel` | character | Hotel type: City or Resort |
| `is_canceled` | integer | 1 = Canceled, 0 = Not Canceled |
| `lead_time` | integer | Days between booking and arrival |
| `arrival_date_year` | integer | Year of arrival |
| `arrival_date_month` | character | Month of arrival |
| `stays_in_weekend_nights` | integer | Nights stayed on weekends |
| `stays_in_week_nights` | integer | Nights stayed on weekdays |
| `adults` | integer | Number of adults |
| `children` | integer | Number of children |
| `babies` | integer | Number of babies |
| `meal` | character | Type of meal booked |
| `country` | character | Country code of origin |
| `market_segment` | character | Booking source (e.g., Direct, Online TA) |
| `distribution_channel` | character | Booking channel used |
| `is_repeated_guest` | integer | 1 = Repeated guest, 0 = New guest |
| `previous_cancellations` | integer | Past booking cancellations |
| `previous_bookings_not_canceled` | integer | Past bookings not canceled |
| `reserved_room_type` | character | Initially reserved room type |
| `assigned_room_type` | character | Room type assigned at check-in |
| `booking_changes` | integer | Number of booking modifications |
| `deposit_type` | character | Deposit type (No Deposit, Non-Refund, etc.) |
| `agent` | character | Agent ID who made the booking |
| `company` | character | Company ID (if booking through company) |
| `days_in_waiting_list` | integer | Days on the waiting list |
| `customer_type` | character | Booking type: Contract, Transient, etc. |
| `adr` | float | Average Daily Rate (price per night) |
| `required_car_parking_spaces` | integer | Requested parking spots |
| `total_of_special_requests` | integer | Number of special requests made |
| `reservation_status` | character | Final status (Canceled, No-Show, Check-Out) |
| `reservation_status_date` | date | Date of the last status update |

This dataset is ideal for classification, segmentation, and trend analysis exercises.


## **1. Load and Prepare the Hotel Booking Dataset**

**Business framing:**  
Your hotel client wants to understand which bookings are most at risk of being canceled. But before modeling, your job is to prepare the data to ensure clean and reliable input.

### Do the following:
- Load the `hotels.csv` file from https://raw.githubusercontent.com/vandanara/UofUtah_IS4487/refs/heads/main/DataSets/hotels.csv
- Run a few summary and descriptive statistics to gain an initial understanding  of the data.
- Perform basic data cleaning (that does not cause data leakage). You can try
  - Remove missing values ((you can only impute missing values after splitting the data into training aand test to avoid data leakage)
  - Encode categorical variables (needed in order to use them in the models).
  - Remove outliers that are big
- Create your `X` (features) and `y` (target = `is_canceled`)
- Split the data into training and test sets (70/30)

### In Your Response:
1.1. How many total rows and columns are in the dataset?

1.2. What types of features (categorical, numerical) are included?

1.3. What steps did you take to clean or prepare the data?


In [11]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import time

# 1. Load the dataset with a robust retry mechanism to prevent ConnectionResetErrors
url = 'https://raw.githubusercontent.com/vandanara/UofUtah_IS4487/refs/heads/main/DataSets/hotels.csv'

for attempt in range(5):
    try:
        df = pd.read_csv(url)
        print("Successfully loaded the dataset!")
        break
    except Exception as e:
        print(f"Attempt {attempt + 1} failed with error: {e}. Retrying in 2 seconds...")
        time.sleep(2)
else:
    raise RuntimeError("Failed to load the dataset after 5 attempts due to network issues.")

# 2. Print initial shape
print("Original dataset shape:", df.shape)

# 3. Data Cleaning
# Drop highly null or identifier-like columns first to avoid discarding too many records
cols_to_drop = ['reservation_status', 'reservation_status_date', 'country', 'agent', 'company']
df_cleaned = df.drop(columns=[col for col in cols_to_drop if col in df.columns])

# Drop remaining missing values after dropping those highly sparse columns
df_cleaned = df_cleaned.dropna().copy()

# Remove major outliers (e.g., negative or extreme ADR values)
df_cleaned = df_cleaned[df_cleaned['adr'] >= 0]
df_cleaned = df_cleaned[df_cleaned['adr'] < 1000]

# Encode categorical variables using Label Encoding
le = LabelEncoder()
categorical_cols = df_cleaned.select_dtypes(include=['object']).columns
for col in categorical_cols:
    df_cleaned[col] = le.fit_transform(df_cleaned[col].astype(str))

# 4. Create X (features) and y (target)
X = df_cleaned.drop(columns=['is_canceled'])
y = df_cleaned['is_canceled']

# 5. Split the data into training and test sets (70/30)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)

print(f"\nCleaned dataset shape: {df_cleaned.shape}")
print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

Successfully loaded the dataset!
Original dataset shape: (119390, 32)

Cleaned dataset shape: (119384, 27)
Training set shape: (83568, 26)
Testing set shape: (35816, 26)


### ✍️ Your Response: 🔧
1.1. The dataset has 119,390 rows and 32 columns.

1.2. Categorical: hotel, meal, country, market_segment, distribution_channel, deposit_type, customer_type.

Numerical: lead_time, adults, children, babies, adr, previous_cancellations, booking_changes.

1.3. I removed rows with missing values and removed some outliers. I also dropped some unnecessary columns and encoded categorical features. Finally, I created X and y and split the data into 70% training and 30% testing.

## **2. Build a Naïve Bayes Model**

**Business framing:**  
Naïve Bayes is a quick, baseline model often used for early testing or simple classification problems.

2.1 Write code to do the following:
- Train a Naïve Bayes classifier on your training data
- Use it to predict on your test data
- Print a classification report and confusion matrix



In [14]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Initialize the Naïve Bayes model
nb_model = GaussianNB()

# Train the model
nb_model.fit(X_train, y_train)

# Predict on the test set
y_pred_nb = nb_model.predict(X_test)

# Print metrics
print("=== Naïve Bayes Model Evaluation ===")
print(f"Accuracy Score: {accuracy_score(y_test, y_pred_nb):.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred_nb))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_nb))

=== Naïve Bayes Model Evaluation ===
Accuracy Score: 0.5519

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.34      0.49     22550
           1       0.45      0.91      0.60     13266

    accuracy                           0.55     35816
   macro avg       0.66      0.63      0.55     35816
weighted avg       0.71      0.55      0.53     35816

Confusion Matrix:
[[ 7683 14867]
 [ 1181 12085]]


### Please answer the following:

2.2. How well does the model perform?  And what metric is best used to judge the performance?

2.3. Where might this model be useful for the hotel (e.g. real-time alerts, operational decisions)?

### ✍️ Your Response: 🔧
2.2. Not very well. The accuracy is 55.19%. Recall is important because we want to find bookings that may be canceled.

2.3. The hotel can use this model to find possible cancellations and contact customers in advance.

## 3. Build a Support Vector Machine (SVM) Model

**Business framing:**  
SVM can model more complex relationships and is useful when customer behavior patterns aren't linear or obvious.

3.1 Write code to do the following:
- Train an SVM classifier (use `linear` kernel)
  - Make predictions and evaluate with classification metrics
  - optionally try some tuning to  improve the model (there are two main hyperparameters C and kernel that you can vary)



In [15]:
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler

# SVMs are highly sensitive to feature scales, so we scale the data first
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Initialize LinearSVC (much faster than SVC(kernel='linear') for large datasets)
svm_model = LinearSVC(random_state=42, max_iter=2000, dual=False)

# Train the model
svm_model.fit(X_train_scaled, y_train)

# Predict on the test set
y_pred_svm = svm_model.predict(X_test_scaled)

# Print evaluation metrics
print("=== Support Vector Machine (LinearSVC) Model Evaluation ===")
print(f"Accuracy Score: {accuracy_score(y_test, y_pred_svm):.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred_svm))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_svm))

=== Support Vector Machine (LinearSVC) Model Evaluation ===
Accuracy Score: 0.7921

Classification Report:
              precision    recall  f1-score   support

           0       0.77      0.96      0.85     22550
           1       0.87      0.52      0.65     13266

    accuracy                           0.79     35816
   macro avg       0.82      0.74      0.75     35816
weighted avg       0.81      0.79      0.78     35816

Confusion Matrix:
[[21539  1011]
 [ 6434  6832]]


### Please answer the following questions:

3.2. How well does the model perform?  And what metric is best used to judge the performance?

3.3. In what business situations could SVM provide better insights than simpler models?

### ✍️ Your Response: 🔧
3.2. The model performs fairly well. The accuracy is 79.21%. Recall is important because we want to find bookings that may be canceled.

3.3. When simple models cannot find patterns in the booking data.

## 4. Build a Neural Network Model

**Business framing:**  
Neural networks are flexible and powerful, though they are harder to explain. They may work well when subtle patterns exist in the data.

4.1. Write code to do the following:

- Build a MLBClassifier model using the neural_network package from sklearn
- Choose a simple architecture (e.g., 2 hidden layers)
- Evaluate accuracy and performance
- optionally try some tuning to improve the model performance (NN has many hyperparameters that you can vary: layers, neurons in each layer, batch size, epochs, dropout, optimizer etc)



In [16]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler

# Neural Networks are sensitive to feature scaling, so we scale the data first
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Initialize the MLPClassifier with a simple architecture (e.g., 2 hidden layers with 32 and 16 neurons)
mlp_model = MLPClassifier(
    hidden_layer_sizes=(32, 16),
    activation='relu',
    solver='adam',
    max_iter=500,
    random_state=42
)

# Train the model
mlp_model.fit(X_train_scaled, y_train)

# Predict on the test set
y_pred_mlp = mlp_model.predict(X_test_scaled)

# Print evaluation metrics
print("=== Neural Network (MLPClassifier) Model Evaluation ===")
print(f"Accuracy Score: {accuracy_score(y_test, y_pred_mlp):.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred_mlp))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_mlp))

=== Neural Network (MLPClassifier) Model Evaluation ===
Accuracy Score: 0.8375

Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.89      0.87     22550
           1       0.81      0.74      0.77     13266

    accuracy                           0.84     35816
   macro avg       0.83      0.82      0.82     35816
weighted avg       0.84      0.84      0.84     35816

Confusion Matrix:
[[20173  2377]
 [ 3444  9822]]


### Please answer the following question:

4.2. How does this model compare to the others?

4.3. Would the business be comfortable using a “black box” model like this? Why or why not?


### ✍️ Your Response: 🔧
4.2. The Neural Network has the highest accuracy (83.75%). It performs better than the other models.

4.3. No, because it is hard to understand how the model makes decisions. It also misses some canceled bookings.

## **5. Compare All Three Models**

5.1 Write code to do the following:
- Print and compare the accuracy of the best Naïve Bayes, SVM, and Neural Network models
- Summarize which model performed best



In [17]:
from sklearn.metrics import accuracy_score

# Calculate accuracies for all models
accuracy_nb = accuracy_score(y_test, y_pred_nb)
accuracy_svm = accuracy_score(y_test, y_pred_svm)
accuracy_mlp = accuracy_score(y_test, y_pred_mlp)

print("=== Model Accuracy Comparison ===")
print(f"Naïve Bayes Accuracy:     {accuracy_nb * 100:.2f}%")
print(f"Support Vector Machine:  {accuracy_svm * 100:.2f}%")
print(f"Neural Network (MLP):    {accuracy_mlp * 100:.2f}%")
print("=================================")

# Summarize the performance
models = {
    "Naïve Bayes": accuracy_nb,
    "Support Vector Machine": accuracy_svm,
    "Neural Network (MLP)": accuracy_mlp
}
best_model = max(models, key=models.get)

print(f"\nThe model with the highest overall accuracy is the {best_model} with {models[best_model] * 100:.2f}% accuracy.")

=== Model Accuracy Comparison ===
Naïve Bayes Accuracy:     55.19%
Support Vector Machine:  79.21%
Neural Network (MLP):    83.75%

The model with the highest overall accuracy is the Neural Network (MLP) with 83.75% accuracy.


### Please answer the following questions:

5.1. Which model had the best overall accuracy, training time, interpretability, and ease of use.

5.2. Would you recommend this model for deployment, and why?

### ✍️ Your Response: 🔧

5.1. The Neural Network had the highest accuracy (83.75%). Naive Bayes is faster and easier to understand.

5.2. Not yet. We should check other results before making decisions.

## **6. Final Business Recommendation**

### In Your Response:
6.1. In 100 words or less, write a short recommendation to hotel management based on your analysis.

Possible info to include:
- Which model do you recommend implementing?
- What business problem does it help solve?
- Are there any risks or limitations?
- What additional data might improve the results in the future?

6.2. How does this relate to your customized learning outcome you created in canvas?


### ✍️ Your Response: 🔧

6.1. 1.Neural Network
     2. Help the hotel find bookings that may be canceled.
     3. The model may miss some cancellations.
     4. More customer information and past cancellation records.

6.2. This lab helped me learn how to clean data, build Naive Bayes, SVM, and Neural Network models, and compare their results. I also learned how to use machine learning to solve a real business problem.

## Submission Instructions
✅ Checklist:
- All code cells run without error
- All markdown responses are complete
- Submit on Canvas as instructed

In [18]:
!jupyter nbconvert --to html "assignment_12_Gong Linlin.ipynb"

[NbConvertApp] Converting notebook assignment_12_Gong Linlin.ipynb to html
[NbConvertApp] Writing 328169 bytes to assignment_12_Gong Linlin.html
